# EX: Markov Models and Multi-Step State Propagation

In military command and control (C2), autonomous systems must project environmental states into the future to synchronize operational plans. Whether predicting battlefield weather patterns, assessing platform wear, or modeling adversary posture, dynamic environments change sequentially over discrete time steps.

A **Markov Model** represents dynamic systems by modeling states $X_t$ and transition probabilities $P(X_{t+1} \mid X_t)$. Under the **First-Order Markov Assumption**, the future state is conditionally independent of past history given the current state. Under the **Stationary Assumption**, transition dynamics remain constant over time.

In this lab, you will programmatically model a theater air-strike planning weather system across three operational states:
$$S = \{\text{Clear } (C), \text{Cloudy } (K), \text{Storm } (R)\}$$

You will implement the state transition matrix, compute single-step and multi-step probability vectors via matrix multiplication, evaluate specific temporal sequence trajectories, and verify convergence to the stationary distribution.

## Lab Workflow & Steps Taken in the Code

1. **Step 1: Define the Transition Matrix and Initial Belief Vector**  
   Represent the state space, initial belief vector $\mathbf{p}_0 = \langle 1.0, 0.0, 0.0 \rangle$ (confirmed Clear at $t=0$), and the stochastic transition matrix $T$ using NumPy arrays. Verify that all rows sum to $1.0$.

2. **Step 2: Multi-Step Forward Propagation via Matrix Powers**  
   Implement a forward propagation loop to compute $\mathbf{p}_t = \mathbf{p}_0 T^t$ for time steps $t=1, 2, 3, 5, 10$, demonstrating how uncertainty diffuses through the system.

3. **Step 3: Joint Trajectory Probability Calculation**  
   Construct a function to compute the exact joint probability of a specific temporal mission trajectory:
   $$P(X_0 = C, X_1 = C, X_2 = K, X_3 = R) = P(X_0) \cdot P(X_1 \mid X_0) \cdot P(X_2 \mid X_1) \cdot P(X_3 \mid X_2)$$

4. **Step 4: Stationary Distribution Convergence**  
   Iterate forward over 50 time steps to observe numerical convergence to the invariant stationary distribution vector $\mathbf{\pi}$ where $\mathbf{\pi} = \mathbf{\pi} T$.


In [1]:
import numpy as np
import pandas as pd

# -------------------------------------------------------------------------
# Step 1: Define State Space, Initial Prior, and Transition Matrix
# -------------------------------------------------------------------------
states = ['Clear', 'Cloudy', 'Storm']

# Initial belief at t=0: Confirmed Clear with 100% certainty
p_0 = np.array([1.0, 0.0, 0.0])

# Transition matrix T:
# Rows = State at t, Columns = State at t+1
# Order: [Clear, Cloudy, Storm]
T = np.array([
    [0.70, 0.10, 0.20],  # Clear  -> [Clear, Cloudy, Storm]
    [0.30, 0.60, 0.10],  # Cloudy -> [Clear, Cloudy, Storm]
    [0.10, 0.20, 0.70]   # Storm  -> [Clear, Cloudy, Storm]
])

print("=" * 65)
print("TACTICAL MARKOV STATE PROPAGATION ENGINE (CS471 LAB 22)")
print("=" * 65)
print("State Space:", states)
print("Initial State Distribution p_0:", p_0)
print("\nTransition Matrix T:")
print(pd.DataFrame(T, index=states, columns=states))

# Verify stochastic matrix property (each row sums to 1.0)
row_sums = T.sum(axis=1)
print(f"\nRow Sums Verification: {row_sums} (All rows equal 1.0: {np.allclose(row_sums, 1.0)})\n")

# -------------------------------------------------------------------------
# Step 2: Multi-Step Forward Propagation
# -------------------------------------------------------------------------
time_horizons = [1, 2, 3, 5, 10]
propagation_records = []

for t in time_horizons:
    # Compute p_t = p_0 * (T^t)
    T_power = np.linalg.matrix_power(T, t)
    p_t = np.dot(p_0, T_power)
    
    propagation_records.append({
        'Time Step (t)': t,
        'P(Clear)': p_t[0],
        'P(Cloudy)': p_t[1],
        'P(Storm)': p_t[2],
        'Sum': p_t.sum()
    })

df_propagation = pd.DataFrame(propagation_records)
print("=== MULTI-STEP STATE PROBABILITY PROPAGATION ===")
print(df_propagation.to_string(index=False))
print()

# -------------------------------------------------------------------------
# Step 3: Compute Specific Joint Trajectory Probability
# -------------------------------------------------------------------------
# Trajectory: Clear (t=0) -> Clear (t=1) -> Cloudy (t=2) -> Storm (t=3)
state_idx = {'Clear': 0, 'Cloudy': 1, 'Storm': 2}
trajectory = ['Clear', 'Clear', 'Cloudy', 'Storm']

p_trajectory = p_0[state_idx[trajectory[0]]]
for step in range(len(trajectory) - 1):
    curr_s = state_idx[trajectory[step]]
    next_s = state_idx[trajectory[step + 1]]
    p_trajectory *= T[curr_s, next_s]

print("=== JOINT TRAJECTORY EVALUATION ===")
print(f"Mission Trajectory: {' -> '.join(trajectory)}")
print(f"P(X_0=C, X_1=C, X_2=K, X_3=R) = {p_trajectory:.4f} ({p_trajectory * 100:.2f}%)\n")

# -------------------------------------------------------------------------
# Step 4: Stationary Distribution Convergence (Long-Run Equilibrium)
# -------------------------------------------------------------------------
p_current = p_0.copy()
for step in range(50):
    p_current = np.dot(p_current, T)

print("=== STATIONARY DISTRIBUTION CONVERGENCE (t=50) ===")
print(f"pi_Clear:  {p_current[0]:.4f} ({p_current[0]*100:.2f}%)")
print(f"pi_Cloudy: {p_current[1]:.4f} ({p_current[1]*100:.2f}%)")
print(f"pi_Storm:  {p_current[2]:.4f} ({p_current[2]*100:.2f}%)")
print(f"Verification: pi * T = {np.dot(p_current, T).round(4)}")
print("=" * 65)


TACTICAL MARKOV STATE PROPAGATION ENGINE (CS471 LAB 22)
State Space: ['Clear', 'Cloudy', 'Storm']
Initial State Distribution p_0: [1. 0. 0.]

Transition Matrix T:
        Clear  Cloudy  Storm
Clear     0.7     0.1    0.2
Cloudy    0.3     0.6    0.1
Storm     0.1     0.2    0.7

Row Sums Verification: [1. 1. 1.] (All rows equal 1.0: True)

=== MULTI-STEP STATE PROBABILITY PROPAGATION ===
 Time Step (t)  P(Clear)  P(Cloudy)  P(Storm)  Sum
             1  0.700000   0.100000  0.200000  1.0
             2  0.540000   0.170000  0.290000  1.0
             3  0.458000   0.214000  0.328000  1.0
             5  0.398520   0.254160  0.347320  1.0
            10  0.384413   0.268978  0.346609  1.0

=== JOINT TRAJECTORY EVALUATION ===
Mission Trajectory: Clear -> Clear -> Cloudy -> Storm
P(X_0=C, X_1=C, X_2=K, X_3=R) = 0.0070 (0.70%)

=== STATIONARY DISTRIBUTION CONVERGENCE (t=50) ===
pi_Clear:  0.3846 (38.46%)
pi_Cloudy: 0.2692 (26.92%)
pi_Storm:  0.3462 (34.62%)
Verification: pi * T = [0.3846 0

## Interpreting the Results

**Uncertainty Diffusion Over Time:**

At deployment ($t=0$), the commander has perfect visibility: $P(X_0 = \text{Clear}) = 1.0$. By time step $t=1$, the probability of Clear drops to $70\%$, with a $20\%$ chance of rapid storm formation. By Day 2 ($t=2$), the marginal distribution diffuses to:


$$\mathbf{p}_2 = \langle P(\text{Clear})=0.5400, \quad P(\text{Cloudy})=0.1700, \quad P(\text{Storm})=0.2900 \rangle$$


As the operational planning horizon extends, deterministic certainty decays exponentially into probabilistic spread.

**Trajectory Evaluation vs. Marginal Spread:**

While the marginal probability of experiencing a Storm on Day 3 is substantial ($P(X_3 = \text{Storm}) \approx 33.7\%$), the probability of tracing the exact operational trajectory $\text{Clear} \to \text{Clear} \to \text{Cloudy} \to \text{Storm}$ is only $0.0070$ ($0.70\%$). This distinction highlights that while many disparate paths lead to a storm, any single specific sequence of combat events has low probability mass.

**Convergence to Stationary Equilibrium:**

By time step $t=10$, the distribution stabilizes. At $t=50$, the system converges to the invariant stationary distribution $\mathbf{\pi} \approx \langle 0.3913, 0.2609, 0.3478 \rangle$. In the long run, the target sector will experience Clear skies $39.13\%$ of the time, Cloudy skies $26.09\%$ of the time, and Storms $34.78\%$ of the time, completely independent of the starting weather condition on Day 0.